# 04. Churn Prediction & Model Evaluation
### Customer Intelligence Platform for Revenue Retention and CLV Optimization

This notebook builds, tunes, and compares three models to predict customer churn,
discusses data leakage prevention, and selects the optimal model for deployment.

---

## 🎯 Business Objective
Retaining an existing customer costs far less than acquiring a new one. To run proactive
win-back campaigns we need a model that flags **which customers are at risk of churning**
before they disengage permanently.

Goals:
- Label customers as churned (inactive > 120 days) vs active.
- Train and compare Logistic Regression (baseline), Random Forest, and XGBoost.
- Select the winner by ROC-AUC and persist it to `models/`.

## 📊 Methodology & Avoiding Data Leakage
The churn label is derived directly from `recency_days > 120`, so **`recency_days` must be
dropped** from the feature matrix before training. Including it would let the model trivially
learn the rule and achieve 100 % accuracy — useful for nothing.

Similarly, raw date columns, cluster labels, and PCA coordinates are excluded.
`customer_city` (~4 000 unique values) is excluded to prevent OHE memory explosion;
`customer_state` (27 values) is kept.

### Setup & Imports

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import validate_config, REPORTS_DIR, FIGURES_DIR, MODELS_DIR
from src.modeling.churn.trainer import train_and_compare, prepare_data

validate_config()
sns.set_theme(style='whitegrid')
print('✓ Environment ready.')

✓ Environment ready.


### 1. Train & Tune All Three Models
We run the training pipeline. Models are compared on a held-out 20 % test set.

In [2]:
results = train_and_compare()

# ── Metrics summary table
metrics_df = pd.DataFrame(
    {name: data['metrics'] for name, data in results.items()}
).T
print('\n=== Model Comparison Table ===')
display(metrics_df.round(4))


=== Model Comparison Table ===


,Accuracy,Precision,Recall,F1 Score,ROC AUC
Logistic Regression,0.7475,0.7500,0.9930,0.8545,0.5864
Random Forest,0.7609,0.7596,0.9947,0.8614,0.7389
XGBoost,0.7981,0.8066,0.9599,0.8766,0.8250


## 🔍 Analysis: ROC Curve Comparison
We rebuild each tuned model from its best parameters (stored in `results`) and plot the
ROC curve on the same test split.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

X, y, preprocessor = prepare_data()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def build_clf(name, params):
    """Reconstruct a classifier from best_params dict (keys are stripped of 'model__')."""
    if name == 'Logistic Regression':
        return LogisticRegression(C=params.get('C', 1.0), max_iter=1000, random_state=42)
    elif name == 'Random Forest':
        return RandomForestClassifier(
            n_estimators=params.get('n_estimators', 100),
            max_depth=params.get('max_depth', 10),
            random_state=42, n_jobs=1
        )
    elif name == 'XGBoost':
        return XGBClassifier(
            n_estimators=params.get('n_estimators', 100),
            max_depth=params.get('max_depth', 6),
            learning_rate=params.get('learning_rate', 0.1),
            random_state=42, eval_metric='logloss', n_jobs=1, verbosity=0
        )

plt.figure(figsize=(9, 7))
for name, data in results.items():
    clf = build_clf(name, data['best_params'])
    pipe = Pipeline([('preprocessor', preprocessor), ('model', clf)])
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{name}  (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison — Churn Prediction')
plt.legend(loc='lower right')
roc_path = FIGURES_DIR / 'churn_roc_comparison.png'
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {roc_path}')

✓ Saved → /Users/asunthalovelin/Documents/P1/reports/figures/churn_roc_comparison.png


/var/folders/8n/qcd0019x7nl4j1_p9k5shwwh0000gn/T/ipykernel_2913/2483094248.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2. Confusion Matrix — Best Model (XGBoost)

In [4]:
best_model = joblib.load(MODELS_DIR / 'churn_model.joblib')
y_pred = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Active (0)', 'Churned (1)'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — XGBoost Churn Model')
cm_path = FIGURES_DIR / 'churn_confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {cm_path}')
print()
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

✓ Saved → /Users/asunthalovelin/Documents/P1/reports/figures/churn_confusion_matrix.png

              precision    recall  f1-score   support

      Active       0.73      0.32      0.45      4807
     Churned       0.81      0.96      0.88     14190

    accuracy                           0.80     18997
   macro avg       0.77      0.64      0.66     18997
weighted avg       0.79      0.80      0.77     18997



/var/folders/8n/qcd0019x7nl4j1_p9k5shwwh0000gn/T/ipykernel_2913/3803297241.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 💡 Business Interpretation & Model Selection Rationale

| Model | Accuracy | Precision | Recall | F1 | **ROC AUC** |
|---|---|---|---|---|---|
| Logistic Regression | 0.7475 | 0.7500 | 0.9930 | 0.8545 | 0.5864 |
| Random Forest | 0.7609 | 0.7596 | 0.9947 | 0.8614 | 0.7389 |
| **XGBoost ✅** | **0.7980** | **0.8077** | **0.9575** | **0.8763** | **0.8225** |

**Why XGBoost wins:**
- Logistic Regression assumes a linear decision boundary — churn drivers such as the
  *interaction* between negative reviews and delivery delays are highly non-linear.
- Random Forest captures non-linearity but is limited by averaging independent trees.
- XGBoost's sequential boosting corrects residual errors at each step, capturing complex
  interactions without overfitting on this dataset size.

**Business trade-off (Recall vs Precision):**
All models achieve very high Recall (>95 %). In retention campaigns this is desirable:
we'd rather spend marketing budget on a few active customers by mistake than miss
a genuinely at-risk customer. XGBoost additionally raises Precision to 0.81,
reducing wasted campaign spend.

## ✅ Key Findings Summary
- **Churn rate: 74.7 %** — consistent with Olist's ~97 % single-order buyer base.
- **XGBoost ROC-AUC: 0.82** — a strong discriminator between active and churned customers.
- **Zero data leakage** — recency_days and all date-derived columns were excluded.

## 🚀 Business Recommendations
- **Threshold tuning:** At 0.5 probability cutoff, deploy retention offers. Lower the threshold
  to 0.35 to run broader win-back campaigns at higher recall.
- **Trigger alerts:** If `avg_review_score` drops below 3 or `delayed_order_count` increases,
  re-score the customer immediately.
- **Segment-specific campaigns:** Combine churn probability with customer segment labels
  (from Phase 3) to tailor campaign messaging.

## 📁 Outputs Generated
- `models/churn_model.joblib` — XGBoost best pipeline
- `models/churn_preprocessor.joblib` — ColumnTransformer
- `reports/churn_metrics.json` — Full metrics for all 3 models
- `reports/figures/churn_roc_comparison.png`
- `reports/figures/churn_confusion_matrix.png`